# Kata 16 — Protocolo de Handoff a Humano

> Spec: [`specs/016-human-handoff/spec.md`](../../specs/016-human-handoff/spec.md)  |  Arquitectura: [`ARCHITECTURE.md`](../../ARCHITECTURE.md)

## Setup

Si `ANTHROPIC_API_KEY` no está en el entorno, se pedirá aquí (sin echo).

In [ ]:
from katas._shared.bootstrap import bootstrap

client, settings = bootstrap(model="claude-sonnet-4-6", budget_calls=8)
print("modelo:", settings.model)

## 1. Concepto en mis palabras

Cuando el agente toca una política que no puede resolver, invoca el tool `escalate_to_human` con un payload tipado: `customer_id`, `issue_summary`, `actions_taken`, `escalation_reason`. La generación de prosa se suspende; el operador recibe un paquete autocontenido.

## 2. Por qué importa

Pasar `messages[]` cruda al humano lo obliga a leer un transcript completo bajo presión. Un payload tipado le da contexto accionable en 30 segundos.

## 3. Ejemplo correcto — tool con schema estricto

In [ ]:
import json

ESCALATE_TOOL = {
    "name": "escalate_to_human",
    "description": "Escala el caso a un operador humano. ÚNICA forma de transferir.",
    "input_schema": {
        "type": "object",
        "properties": {
            "customer_id": {"type": "string"},
            "issue_summary": {"type": "string", "minLength": 10},
            "actions_taken": {"type": "array", "items": {"type": "string"}, "minItems": 1},
            "escalation_reason": {
                "type": "string",
                "enum": ["policy_limit","data_conflict","missing_info","customer_demand","other"],
            },
            "recommended_action": {"type": "string"},
        },
        "required": ["customer_id","issue_summary","actions_taken","escalation_reason","recommended_action"],
    },
}

REFUND_TOOL = {
    "name": "process_refund",
    "description": "Procesa reembolsos hasta $1000. Para más, escala.",
    "input_schema": {
        "type": "object",
        "properties": {"customer_id":{"type":"string"},"amount":{"type":"number"}},
        "required": ["customer_id","amount"],
    },
}

system = (
    "Eres un agente de soporte. Reembolsos hasta $1000 los procesas con process_refund. "
    "Si el cliente pide más de $1000, llama a escalate_to_human con el payload completo. "
    "NUNCA produzcas el handoff en prosa."
)
user_req = "Soy el cliente VIP-007. Necesito un reembolso de $5000 por una compra defectuosa que reporté tres veces. Tengo el ticket T-123."

resp = client.messages.create(
    system=system,
    messages=[{"role":"user","content": user_req}],
    tools=[REFUND_TOOL, ESCALATE_TOOL],
)

for b in resp.content:
    if b.type == "tool_use":
        print(f"tool: {b.name}")
        print(json.dumps(b.input, indent=2, ensure_ascii=False))
    elif b.type == "text":
        print("texto:", b.text)
print("stop_reason:", resp.stop_reason)

### 3.1 Validar shape antes de despachar al operador

In [ ]:
def dispatch_to_operator(handoff: dict):
    required = ESCALATE_TOOL["input_schema"]["required"]
    missing = [k for k in required if k not in handoff]
    assert not missing, f"handoff incompleto, faltan: {missing}"
    assert len(handoff["issue_summary"]) >= 10, "summary demasiado corto"
    assert handoff["escalation_reason"] in ESCALATE_TOOL["input_schema"]["properties"]["escalation_reason"]["enum"]
    print("OK: handoff válido, despachando a la cola humana")
    print(handoff)

handoff = next((b.input for b in resp.content if b.type == "tool_use" and b.name == "escalate_to_human"), None)
if handoff:
    dispatch_to_operator(handoff)
else:
    print("(el modelo no escaló; revisa system prompt)")

## 4. Anti-patrón — handoff en prosa libre

In [ ]:
system_prose = (
    "Eres un agente de soporte. Si la solicitud excede política, redacta un email amigable "
    "explicando al humano todo el contexto."
)
resp_bad = client.messages.create(
    system=system_prose,
    messages=[{"role":"user","content": user_req}],
)
print(next((b.text for b in resp_bad.content if b.type == "text"), "")[:600])

Problemas observables:

- El humano debe leer prosa para extraer customer_id, monto, acciones tomadas — datos que ya estaban estructurados en el contexto.
- No hay enum de `reason`: el operador no puede agrupar/priorizar la cola por motivo.
- El campo `actions_taken` queda al criterio del modelo; puede inventarlo.

El handoff tipado no admite ninguna de esas ambigüedades.

## 5. Argumento de certificación

- **`escalate_to_human` es un tool, no una intención.** El modelo lo invoca o no, pero su llamada produce JSON válido o falla cerrada.
- **`actions_taken` con `minItems: 1`**: el humano sabe qué se intentó.
- **`escalation_reason` con enum**: la cola humana se puede agrupar por motivo.
- **Conexión con otros katas**: el `needs_human_review` del Kata 15 dispara este handoff; el conflicto de provenance del Kata 20 también; los hooks Kata 2 lo invocan en `ask_human`.

## 6. Auto-evaluación

**1. ¿Qué hago si el modelo intenta seguir generando prosa después del tool call?**

El bucle del cliente, al detectar `tool_use=escalate_to_human`, despacha al humano y termina la sesión. No se inyecta ningún `tool_result` que reanude la conversación: el handoff es un end-state, no una pausa.

**2. ¿Cómo aseguro que `actions_taken` refleja realmente lo ejecutado y no alucinación?**

El cliente mantiene un log canónico de las acciones que se ejecutaron en el bucle (los `tool_use` aceptados con sus resultados). En el dispatcher, comparo `handoff["actions_taken"]` contra ese log y rechazo el handoff si menciona una acción que nunca corrió.

**3. ¿Cuándo es legítimo no escalar y simplemente abortar?**

Cuando el costo de resolver excede el valor del caso (peticiones spam, abuso). Aborto = respuesta fija "no procedemos" sin invocar al humano. Debe registrarse para que el operador, si revisa la cola de abortos por muestreo, detecte falsos positivos.